# 🎵 Mi Spotify Wrapped — EDA Personal
**Universidad de Pamplona · Bases de Datos II · 2026-I**

Análisis exploratorio sobre datos reales extraídos de mi cuenta de Spotify
y almacenados en el Data Warehouse (PostgreSQL Neon).

Pipeline: `Spotify Web API → FastAPI ETL → dim_artists / dim_tracks / fact_listening_history`


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os
import warnings
warnings.filterwarnings("ignore")

load_dotenv()

# ── Conexión a Neon ────────────────────────────────────────────────────────
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# Paleta Spotify
SPOTIFY_GREEN = "#1DB954"
DARK_BG       = "#121212"
CARD_BG       = "#1E1E1E"
TEXT_COLOR     = "#FFFFFF"

plt.rcParams.update({
    "figure.facecolor":  DARK_BG,
    "axes.facecolor":    CARD_BG,
    "axes.edgecolor":    "#333333",
    "axes.labelcolor":   TEXT_COLOR,
    "xtick.color":       TEXT_COLOR,
    "ytick.color":       TEXT_COLOR,
    "text.color":        TEXT_COLOR,
    "grid.color":        "#2a2a2a",
    "grid.linestyle":    "--",
    "font.family":       "DejaVu Sans",
})

print("✅ Conexión lista")


In [ ]:
# ── 1. Top artistas con play count real ───────────────────────────────────
df_artists = pd.read_sql(text("""
    SELECT
        a.name,
        a.popularity,
        a.followers_count,
        a.genres,
        COUNT(f.id) AS play_count
    FROM dwh.dim_artists a
    JOIN dwh.fact_listening_history f ON f.artist_id = a.artist_id
    JOIN dwh.dim_users u ON u.user_id = f.user_id
    GROUP BY a.artist_id, a.name, a.popularity, a.followers_count, a.genres
    ORDER BY play_count DESC
    LIMIT 20
"""), engine)

# ── 2. Actividad por hora ──────────────────────────────────────────────────
df_hours = pd.read_sql(text("""
    SELECT hour_of_day, COUNT(*) AS plays
    FROM dwh.fact_listening_history
    GROUP BY hour_of_day
    ORDER BY hour_of_day
"""), engine)

# ── 3. Top tracks ──────────────────────────────────────────────────────────
df_tracks = pd.read_sql(text("""
    SELECT
        t.name,
        a.name AS artist,
        t.popularity,
        t.duration_ms,
        COUNT(f.id) AS play_count
    FROM dwh.dim_tracks t
    JOIN dwh.dim_artists a ON a.artist_id = t.artist_id
    JOIN dwh.fact_listening_history f ON f.track_id = t.track_id
    GROUP BY t.track_id, t.name, a.name, t.popularity, t.duration_ms
    ORDER BY play_count DESC
    LIMIT 20
"""), engine)

# ── 4. Actividad por día de la semana ─────────────────────────────────────
DAY_MAP = {"Monday":0,"Tuesday":1,"Wednesday":2,"Thursday":3,
           "Friday":4,"Saturday":5,"Sunday":6}
DAY_LABELS = ["Lun","Mar","Mié","Jue","Vie","Sáb","Dom"]

df_days = pd.read_sql(text("""
    SELECT day_of_week, COUNT(*) AS plays
    FROM dwh.fact_listening_history
    GROUP BY day_of_week
"""), engine)
df_days["day_num"] = df_days["day_of_week"].map(DAY_MAP)
df_days = df_days.sort_values("day_num")

# ── 5. Géneros ─────────────────────────────────────────────────────────────
df_genres = pd.read_sql(text("""
    SELECT UNNEST(a.genres) AS genre, COUNT(DISTINCT a.artist_id) AS artist_count
    FROM dwh.dim_artists a
    JOIN dwh.fact_listening_history f ON f.artist_id = a.artist_id
    JOIN dwh.dim_users u ON u.user_id = f.user_id
    WHERE a.genres IS NOT NULL AND array_length(a.genres,1) > 0
    GROUP BY genre
    ORDER BY artist_count DESC
    LIMIT 15
"""), engine)

print(f"✅ Artistas cargados:      {len(df_artists)}")
print(f"✅ Tracks cargados:        {len(df_tracks)}")
print(f"✅ Horas con actividad:    {len(df_hours)}")
print(f"✅ Días con actividad:     {len(df_days)}")
print(f"✅ Géneros únicos (top15): {len(df_genres)}")


## 📊 Vista general del DWH

In [ ]:
with engine.connect() as conn:
    total_plays   = conn.execute(text("SELECT COUNT(*) FROM dwh.fact_listening_history")).scalar()
    total_artists = conn.execute(text("SELECT COUNT(*) FROM dwh.dim_artists")).scalar()
    total_tracks  = conn.execute(text("SELECT COUNT(*) FROM dwh.dim_tracks")).scalar()
    peak_hour_row = conn.execute(text("""
        SELECT hour_of_day, COUNT(*) AS plays
        FROM dwh.fact_listening_history
        GROUP BY hour_of_day ORDER BY plays DESC LIMIT 1
    """)).fetchone()

peak_hour  = peak_hour_row[0]
peak_plays = peak_hour_row[1]

print("=" * 45)
print(f"  {'ESTADÍSTICAS DEL DWH':^43}")
print("=" * 45)
print(f"  Reproducciones totales:   {total_plays:>8,}")
print(f"  Artistas únicos:          {total_artists:>8,}")
print(f"  Canciones únicas:         {total_tracks:>8,}")
print(f"  Hora pico:                {peak_hour:>7}:00 hs  ({peak_plays} plays)")
print(f"  Avg duración (tracks):    {df_tracks['duration_ms'].mean()/1000/60:>7.2f} min")
print(f"  Avg popularidad (tracks): {df_tracks['popularity'].mean():>7.1f} / 100")
print("=" * 45)


## 📈 Gráfico 1 — Actividad por hora del día
### 💡 El gráfico que más me sorprendió

Esperaba que mi pico de escucha fuera en la noche. El resultado fue completamente diferente.


In [ ]:
# Rellenar horas sin actividad con 0
all_hours = pd.DataFrame({"hour_of_day": range(24)})
df_hours_full = all_hours.merge(df_hours, on="hour_of_day", how="left").fillna(0)
df_hours_full["plays"] = df_hours_full["plays"].astype(int)

peak_h = df_hours_full.loc[df_hours_full["plays"].idxmax(), "hour_of_day"]

fig, ax = plt.subplots(figsize=(14, 5))

colors = [SPOTIFY_GREEN if h == peak_h else "#444444" for h in df_hours_full["hour_of_day"]]
bars = ax.bar(df_hours_full["hour_of_day"], df_hours_full["plays"], color=colors, width=0.7, zorder=3)

# Anotar la barra pico
max_plays = df_hours_full["plays"].max()
ax.annotate(
    f"Pico: {int(peak_h):02d}:00 hs\n{int(max_plays)} plays",
    xy=(peak_h, max_plays),
    xytext=(peak_h + 1.5, max_plays - 1),
    fontsize=10, color=SPOTIFY_GREEN, fontweight="bold",
    arrowprops=dict(arrowstyle="->", color=SPOTIFY_GREEN),
)

ax.set_xlabel("Hora del día", fontsize=11)
ax.set_ylabel("Reproducciones", fontsize=11)
ax.set_title("¿A qué hora escucho más música?", fontsize=14, fontweight="bold", pad=15)
ax.set_xticks(range(24))
ax.set_xticklabels([f"{h:02d}:00" for h in range(24)], rotation=45, ha="right", fontsize=8)
ax.grid(axis="y", zorder=0)
ax.set_xlim(-0.5, 23.5)

fig.tight_layout()
plt.savefig("docs/assets/eda_hora_pico.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

print(f"\n🔍 Mi hora pico es las {int(peak_h):02d}:00 hs con {int(max_plays)} reproducciones.")
print(f"   Esto me sorprendió porque esperaba escuchar más música de noche.")
print(f"   En cambio, mi pico ocurre a las {int(peak_h):02d}:00 — que corresponde a horario de mañana/trabajo.")


## 🎤 Gráfico 2 — Mis artistas más escuchados

In [ ]:
top10 = df_artists.head(10).sort_values("play_count")

fig, ax = plt.subplots(figsize=(12, 6))

bars = ax.barh(top10["name"], top10["play_count"], color=SPOTIFY_GREEN, height=0.6)

for bar, val in zip(bars, top10["play_count"]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(int(val)), va="center", fontsize=10, color=TEXT_COLOR, fontweight="bold")

ax.set_xlabel("Reproducciones en el DWH", fontsize=11)
ax.set_title("Top 10 artistas más escuchados", fontsize=14, fontweight="bold", pad=15)
ax.set_xlim(0, top10["play_count"].max() * 1.2)
ax.grid(axis="x", zorder=0)

fig.tight_layout()
plt.savefig("docs/assets/eda_top_artistas.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()


## 🎸 Gráfico 3 — Géneros dominantes

In [ ]:
top_genres = df_genres.head(10)

palette = sns.color_palette("Greens_r", len(top_genres))

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.barh(top_genres["genre"][::-1], top_genres["artist_count"][::-1],
               color=palette[::-1], height=0.6)

for bar, val in zip(bars, top_genres["artist_count"][::-1]):
    ax.text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
            str(int(val)), va="center", fontsize=10, color=TEXT_COLOR)

ax.set_xlabel("Número de artistas", fontsize=11)
ax.set_title("Géneros más presentes en mi historial", fontsize=14, fontweight="bold", pad=15)
ax.grid(axis="x", zorder=0)

fig.tight_layout()
plt.savefig("docs/assets/eda_generos.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()


## 📅 Gráfico 4 — Actividad por día de la semana

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors_day = [SPOTIFY_GREEN if v == df_days["plays"].max() else "#444444"
              for v in df_days["plays"]]

ax.bar(range(len(df_days)), df_days["plays"], color=colors_day, width=0.6, zorder=3)
ax.set_xticks(range(len(df_days)))
ax.set_xticklabels(DAY_LABELS[:len(df_days)], fontsize=12)
ax.set_ylabel("Reproducciones", fontsize=11)
ax.set_title("¿Qué día de la semana escucho más?", fontsize=14, fontweight="bold", pad=15)
ax.grid(axis="y", zorder=0)

for i, v in enumerate(df_days["plays"]):
    ax.text(i, v + 0.2, str(int(v)), ha="center", fontsize=11,
            color=SPOTIFY_GREEN if v == df_days["plays"].max() else TEXT_COLOR,
            fontweight="bold")

fig.tight_layout()
plt.savefig("docs/assets/eda_dias.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()


## 🔬 Gráfico 5 — ¿Escucho artistas populares o nichos?
### 💡 Qué aprendí de mis propios datos que no sabía antes


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    df_artists["popularity"],
    df_artists["play_count"],
    s=df_artists["followers_count"] / df_artists["followers_count"].max() * 800 + 50,
    c=df_artists["play_count"],
    cmap="Greens",
    alpha=0.85,
    edgecolors="#ffffff",
    linewidths=0.5,
    zorder=3,
)

# Etiquetar puntos
for _, row in df_artists.head(10).iterrows():
    ax.annotate(
        row["name"],
        (row["popularity"], row["play_count"]),
        textcoords="offset points",
        xytext=(8, 3),
        fontsize=8,
        color=TEXT_COLOR,
        alpha=0.9,
    )

ax.axvline(x=50, color="#555555", linestyle="--", linewidth=1, label="Umbral popularidad media (50)")
ax.set_xlabel("Popularidad en Spotify (0–100)", fontsize=11)
ax.set_ylabel("Reproducciones en mi historial", fontsize=11)
ax.set_title("Popularidad vs. mis reproducciones\n(tamaño = followers del artista)", fontsize=13, fontweight="bold", pad=15)
ax.legend(fontsize=9)
ax.grid(zorder=0)

plt.colorbar(scatter, ax=ax, label="Play count")
fig.tight_layout()
plt.savefig("docs/assets/eda_popularidad_vs_plays.png", dpi=150, bbox_inches="tight", facecolor=DARK_BG)
plt.show()

low_pop = df_artists[df_artists["popularity"] < 10]
high_pop = df_artists[df_artists["popularity"] >= 50]
print(f"Artistas con popularidad < 10:  {len(low_pop)} ({len(low_pop)/len(df_artists)*100:.0f}%)")
print(f"Artistas con popularidad >= 50: {len(high_pop)} ({len(high_pop)/len(df_artists)*100:.0f}%)")
print(f"\n💡 Descubrimiento: la mayoría de mis artistas tienen popularidad < 10 en Spotify,")
print(f"   lo que indica un perfil de escucha muy orientado a música latina de nicho")
print(f"   (reggaeton underground, vallenato) más que a pop mainstream.")


## 🔎 Consultas analíticas del parcial

In [ ]:
print("=" * 50)
print("Pregunta 1 — ¿En qué hora del día escuchas más?")
print("=" * 50)
q1 = pd.read_sql(text("""
    SELECT hour_of_day, COUNT(*) AS reproducciones
    FROM dwh.fact_listening_history
    GROUP BY hour_of_day
    ORDER BY reproducciones DESC
    LIMIT 5
"""), engine)
print(q1.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 2 — ¿Cuál es tu artista más escuchado?")
print("=" * 50)
q2 = pd.read_sql(text("""
    SELECT a.name, COUNT(*) AS veces
    FROM dwh.fact_listening_history f
    JOIN dwh.dim_artists a ON a.artist_id = f.artist_id
    GROUP BY a.name
    ORDER BY veces DESC
    LIMIT 5
"""), engine)
print(q2.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 3 — ¿Cuál es tu canción más repetida?")
print("=" * 50)
q3 = pd.read_sql(text("""
    SELECT t.name, a.name AS artista, COUNT(*) AS veces
    FROM dwh.fact_listening_history f
    JOIN dwh.dim_tracks t ON t.track_id = f.track_id
    JOIN dwh.dim_artists a ON a.artist_id = f.artist_id
    GROUP BY t.name, a.name
    ORDER BY veces DESC
    LIMIT 5
"""), engine)
print(q3.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 4 — ¿Qué género domina tu historial?")
print("=" * 50)
q4 = pd.read_sql(text("""
    SELECT UNNEST(a.genres) AS genero, COUNT(*) AS apariciones
    FROM dwh.dim_artists a
    JOIN dwh.fact_listening_history f ON f.artist_id = a.artist_id
    JOIN dwh.dim_users u ON u.user_id = f.user_id
    WHERE array_length(a.genres,1) > 0
    GROUP BY genero
    ORDER BY apariciones DESC
    LIMIT 5
"""), engine)
print(q4.to_string(index=False))

print("\n" + "=" * 50)
print("Pregunta 5 — ¿En qué día escuchas más música?")
print("=" * 50)
q5 = pd.read_sql(text("""
    SELECT day_of_week, COUNT(*) AS reproducciones
    FROM dwh.fact_listening_history
    GROUP BY day_of_week
    ORDER BY reproducciones DESC
    LIMIT 5
"""), engine)
print(q5.to_string(index=False))


## 💬 Reflexiones finales

### 1. El gráfico que más me sorprendió
El gráfico de **actividad por hora del día** fue el más inesperado.
Mi hipótesis era que escucharía más música de noche (20:00–23:00),
pero los datos muestran un pico claro a las **11:00 hs** con 16 reproducciones,
seguido de las 10:00 hs. Esto revela que escucho música principalmente
durante el horario de la mañana — posiblemente mientras estudio o trabajo —
no como actividad nocturna de entretenimiento.

---

### 2. Qué aprendí de mis propios datos que no sabía antes
No sabía que mi catálogo personal está tan concentrado en **reggaeton
latinoamericano de nicho** (artistas con popularidad < 10 en Spotify).
De mis top artistas, solo **The Smiths** supera popularidad 50.
El resto son artistas puertorriqueños y colombianos con millones de
followers en Last.fm pero baja popularidad en el índice de Spotify,
lo que muestra una discrepancia entre popularidad global (Spotify) y
popularidad regional/cultural real.

---

### 3. Qué pregunta quise hacerle a los datos y no pude

**¿Cómo evoluciona mi gusto musical a lo largo del tiempo?**

El modelo actual no lo permite porque `fact_listening_history` solo guarda
las últimas 50 reproducciones por limitación de `GET /v1/me/player/recently-played`.
No hay datos históricos de meses o años anteriores.

Para responder esta pregunta habría que agregar al modelo:

```sql
-- Nueva columna en fact_listening_history:
week_number  INT,   -- semana del año (1–52)
month        INT,   -- mes (1–12)
year         INT    -- año

-- Nueva tabla de dimensión temporal:
CREATE TABLE dwh.dim_time (
    time_id     SERIAL PRIMARY KEY,
    date        DATE,
    week        INT,
    month       INT,
    year        INT,
    quarter     INT,
    day_type    VARCHAR(10) -- 'weekday' / 'weekend'
);
```

Con esto se podría responder: *¿en qué mes del año escucho más reggaeton?*,
*¿mi gusto cambió después del verano?*, *¿hay canciones que solo escucho
ciertos meses?*

También requeriría un mecanismo de **scrobbling continuo** (escuchar en
tiempo real y guardar cada reproducción), no solo pulls periódicos de las
últimas 50.
